## Data inladen

In [ ]:
pip install pandas openpyxl xlrd scikit-learn scipy seaborn matplotlib

In [ ]:
import pandas as pd

df = pd.read_excel("mobiliteitsdata.xlsx", engine="openpyxl")
df.head()

## Beschrijvende statistieken

In [ ]:
df.describe()

## Missende, niet-kloppende en extreme waarden

We checken eerst hoeveel missende waarden en hoeveel onmogelijke (negatieve) waarden er per kolom zijn.

In [ ]:
print("Missende waarden per kolom:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nNegatieve waarden per kolom:")
for kolom in df.select_dtypes(include="number").columns:
    aantal_negatief = (df[kolom] < 0).sum()
    if aantal_negatief > 0:
        print(f"{kolom}: {aantal_negatief} negatieve waarden")

We filteren de rijen met negatieve waarden voor jaarsalaris, autokilometers en OV-kilometers eruit. Omdat een vergelijking met een missende waarde (NaN) altijd `False` oplevert, valt de ene rij met een missend jaarsalaris hier automatisch ook uit.

In [ ]:
new_df = df[
    (df["jaarsalaris_eu"] >= 0) &
    (df["km_auto_per_jaar"] >= 0) &
    (df["km_ov_per_jaar"] >= 0)
]

new_df.shape

## Transformatie van de variabelen

We laten `persoon_ID` buiten beschouwing, want dat is alleen een identificatienummer en zegt statistisch niets. Sterk scheve numerieke kolommen transformeren we met een log-transformatie, daarna schalen we alle numerieke kolommen en zetten we de categorische kolommen om met one-hot encoding. De originele dataset (`new_df`) blijft ongewijzigd staan.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

transformed_df = new_df.drop(columns=["persoon_ID"]).copy()

num_cols = transformed_df.select_dtypes(include="number").columns.tolist()
cat_cols = transformed_df.select_dtypes(include="object").columns.tolist()

# Scheve kolommen (|skewness| > 1) log-transformeren
skewness = transformed_df[num_cols].skew().abs()
skew_cols = skewness[skewness > 1].index.tolist()

for col in skew_cols:
    transformed_df[col + "_log"] = np.log1p(transformed_df[col])
    transformed_df.drop(columns=[col], inplace=True)

# Alle numerieke kolommen schalen
num_cols = transformed_df.select_dtypes(include="number").columns.tolist()
scaler = StandardScaler()
transformed_df[num_cols] = scaler.fit_transform(transformed_df[num_cols])

# Categorische kolommen one-hot encoderen
transformed_df = pd.get_dummies(transformed_df, columns=cat_cols, drop_first=True)

transformed_df.head()

## Relaties tussen de variabelen

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

num_cols = transformed_df.select_dtypes(include="number").columns.tolist()
corr = transformed_df[num_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlatiematrix")
plt.show()

corr_abs = corr.abs()
pairs = []
for i in range(len(num_cols)):
    for j in range(i + 1, len(num_cols)):
        pairs.append((num_cols[i], num_cols[j], corr_abs.iloc[i, j]))

pairs_sorted = sorted(pairs, key=lambda x: -x[2])

print("Top 10 sterkste correlaties:")
for a, b, val in pairs_sorted[:10]:
    print(f"{a} <-> {b}: {val:.3f}")

## Kans op een CO2-uitstoot van meer dan 5000 kg

In [ ]:
from scipy.stats import norm

co2 = df["co2_uitstoot_per_jaar_KG"].dropna()
mu = co2.mean()
sigma = co2.std(ddof=0)

z = (5000 - mu) / sigma
kans = 1 - norm.cdf(z)

print(f"Gemiddelde CO2-uitstoot: {mu:.1f} kg")
print(f"Standaardafwijking: {sigma:.1f} kg")
print(f"Z-score voor 5000 kg: {z:.2f}")
print(f"Kans op een CO2-uitstoot van meer dan 5000 kg: {kans:.4f} ({kans * 100:.2f}%)")

## Verschil in CO2-uitstoot: elektrische auto vs. geen elektrische auto

Gekozen groepsvariabele: `elektrisch_auto`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=new_df, x="elektrisch_auto", y="co2_uitstoot_per_jaar_KG", ax=axes[0])
axes[0].set_title("CO2-uitstoot per groep")
axes[0].set_xlabel("Elektrische auto")
axes[0].set_ylabel("CO2-uitstoot per jaar (kg)")

sns.histplot(data=new_df, x="co2_uitstoot_per_jaar_KG", hue="elektrisch_auto", bins=30, kde=True, ax=axes[1])
axes[1].set_title("Verdeling CO2-uitstoot per groep")
axes[1].set_xlabel("CO2-uitstoot per jaar (kg)")

plt.tight_layout()
plt.show()

new_df.groupby("elektrisch_auto")["co2_uitstoot_per_jaar_KG"].describe()

In [ ]:
from scipy import stats

groep_ja = new_df[new_df["elektrisch_auto"] == "ja"]["co2_uitstoot_per_jaar_KG"].dropna()
groep_nee = new_df[new_df["elektrisch_auto"] == "nee"]["co2_uitstoot_per_jaar_KG"].dropna()

stat, p_waarde = stats.mannwhitneyu(groep_ja, groep_nee, alternative="two-sided")

print(f"Mann-Whitney U-statistiek: {stat}")
print(f"p-waarde: {p_waarde:.4g}")

## Lineair regressiemodel: CO2-uitstoot voorspellen

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = new_df.drop(columns=["co2_uitstoot_per_jaar_KG"])
y = new_df["co2_uitstoot_per_jaar_KG"]

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(include="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), categorical_cols)
], remainder="passthrough")

pipeline_all = Pipeline([
    ("preproc", preprocessor),
    ("model", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline_all.fit(X_train, y_train)
y_pred = pipeline_all.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Training set: {X_train.shape[0]} rijen, test set: {X_test.shape[0]} rijen")
print(f"RMSE: {rmse:.2f} kg")
print(f"MAE: {mae:.2f} kg")
print(f"R²: {r2:.3f}")

feature_names = pipeline_all.named_steps["preproc"].get_feature_names_out()
coeff_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": pipeline_all.named_steps["model"].coef_
}).sort_values(by="coefficient", key=lambda col: col.abs(), ascending=False)

print("\nTop 5 onafhankelijke variabelen met de meeste invloed:")
coeff_df.head(5)

## Top 5 variabelen en modelvergelijking (20-fold cross-validation)

In [ ]:
from sklearn.model_selection import cross_val_score

top_5_features_encoded = coeff_df.head(5)["feature"].tolist()

top_5_features_original = []
for feat in top_5_features_encoded:
    naam = feat.replace("cat__", "").rsplit("_", 1)[0] if feat.startswith("cat__") else feat
    if naam not in top_5_features_original:
        top_5_features_original.append(naam)
top_5_features_original = top_5_features_original[:5]

print("Top 5 originele variabelen:", top_5_features_original)

# Model 1: alle variabelen
cv_scores_all = cross_val_score(pipeline_all, X, y, cv=20, scoring="r2")

# Model 2: top 5 variabelen
X_top5 = X[top_5_features_original]
top5_cat = [c for c in top_5_features_original if c in categorical_cols]

preprocessor_top5 = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), top5_cat)
], remainder="passthrough")

pipeline_top5 = Pipeline([
    ("preproc", preprocessor_top5),
    ("model", LinearRegression())
])

cv_scores_top5 = cross_val_score(pipeline_top5, X_top5, y, cv=20, scoring="r2")

print(f"\nModel 1 (alle variabelen)  - gemiddelde R²: {cv_scores_all.mean():.4f}")
print(f"Model 2 (top 5 variabelen) - gemiddelde R²: {cv_scores_top5.mean():.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot([cv_scores_all, cv_scores_top5], tick_labels=["Alle variabelen", "Top 5 variabelen"])
plt.ylabel("R² (20-fold CV)")
plt.title("Vergelijking nauwkeurigheid: alle variabelen vs. top 5")
plt.show()

## Is het verschil tussen de modellen significant?

In [ ]:
verschil = np.array(cv_scores_all) - np.array(cv_scores_top5)

sh_stat, sh_p = stats.shapiro(verschil)
print(f"Shapiro-Wilk test op de verschillen: p = {sh_p:.4f}")

if sh_p > 0.05:
    print("Verschillen zijn normaal verdeeld -> paired t-test")
    stat_val, p_waarde = stats.ttest_rel(cv_scores_all, cv_scores_top5)
else:
    print("Verschillen zijn niet normaal verdeeld -> Wilcoxon signed-rank test")
    stat_val, p_waarde = stats.wilcoxon(cv_scores_all, cv_scores_top5)

print(f"Teststatistiek: {stat_val:.4f}")
print(f"p-waarde: {p_waarde:.4g}")

if p_waarde < 0.05:
    print("Conclusie: het verschil tussen de twee modellen is statistisch significant.")
else:
    print("Conclusie: er is geen statistisch significant verschil tussen de twee modellen.")

## CO2-uitstoot omzetten naar klassen (laag / hoog)

De afkapwaarde is de mediaan, zodat de twee groepen ongeveer even groot zijn.

In [ ]:
new_df = new_df.copy()
new_df["co2_class"] = (new_df["co2_uitstoot_per_jaar_KG"] > new_df["co2_uitstoot_per_jaar_KG"].median()).astype(int)

print(new_df["co2_class"].value_counts())

X_clf = new_df.drop(columns=["co2_uitstoot_per_jaar_KG", "co2_class"])
y_clf = new_df["co2_class"]

## Classificatiemodel met de top 5 variabelen

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

X_top5_clf = X_clf[top_5_features_original]
y = y_clf

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_top5_clf, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols_c = [c for c in top_5_features_original if c in categorical_cols]
num_cols_c = [c for c in top_5_features_original if c in numeric_cols]

preproc_clf = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), cat_cols_c),
    ("num", StandardScaler(), num_cols_c)
], remainder="passthrough")

pipeline_clf = Pipeline([
    ("preproc", preproc_clf),
    ("clf", LogisticRegression(max_iter=1000))
])

pipeline_clf.fit(X_train_c, y_train_c)
y_pred = pipeline_clf.predict(X_test_c)

print(f"Accuracy : {accuracy_score(y_test_c, y_pred):.3f}")
print(f"Precision: {precision_score(y_test_c, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test_c, y_pred):.3f}")
print(f"F1-score : {f1_score(y_test_c, y_pred):.3f}")
print()
print(classification_report(y_test_c, y_pred, digits=3))

ConfusionMatrixDisplay.from_estimator(pipeline_clf, X_test_c, y_test_c, cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()